# 002_naive_eval.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
from IPython.display import display
from nuteval import (
    MODELS,
    CachedRun,
    run_evaluation_for_model,
    load_dataset,
    build_evaluation_dataframe,
    add_error_metrics,
    compute_summary,
    format_summary_for_display,
    plot_mean_ape_per_model_and_nutrient,
    plot_error_distribution_per_model,
    plot_pred_vs_gt_by_nutrient,
    plot_cost_vs_accuracy,
)


## Load test dataset

In [ ]:
meals = load_dataset()

## Run model evaluation

In [ ]:
# test all available models
models_to_evaluate = list(MODELS.keys())
runs_per_meal = 4

all_runs: dict[str, list[CachedRun]] = {}
for model_name in models_to_evaluate:
    print(f"\nEvaluating {model_name}")
    all_runs[model_name] = run_evaluation_for_model(
        model_name=model_name,
        meals=meals,
        runs_per_meal=runs_per_meal
    )
    time.sleep(1)


## Data Prep & Metric Calculation

Transform raw API outputs into a structured evaluation table, calculating MAPE, absolute errors and summarizing data per model.

In [ ]:
df = build_evaluation_dataframe(meals, all_runs)
df = add_error_metrics(df)

summary_df = compute_summary(df)

display(df[["model_name", "meal_id", "calories_ape", "cost_usd", "latency_s"]].head())

## Accuracy

In [ ]:
fig = plot_mean_ape_per_model_and_nutrient(df)
fig.show()

In [ ]:
fig = plot_error_distribution_per_model(df)
fig.show()

In [ ]:
fig = plot_pred_vs_gt_by_nutrient(df)
fig.show()

## Cost and latency

In [ ]:
fig = plot_cost_vs_accuracy(summary_df)
fig.show()

## Summary

In [ ]:
d_summary_df = format_summary_for_display(summary_df)
display(d_summary_df)

### Key Takeaways & Model Selection

* **Top Performer (Accuracy & Cost):** `gemini-3.8-flash` ...
* **Speed / Budget Option:** `gemini-3.5-flash-lite` ...